# Prophet予測精度分析（東京23区）

区ごとにProphetモデルを学習し、2005-2023年のデータでトレーニング、2024-2025年のデータで予測精度を確認します。


In [1]:
# ライブラリのインポート
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from prophet import Prophet
import pandas as pd
import numpy as np

/Users/yamaguchiteppei/opt/post/weekly_dev/clip_automation_poc/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. データ読み込み


In [2]:
# データ読み込み
df = pl.read_csv("../data/ml_dataset/tokyo_23_ml_dataset.csv")

# transaction_dateをDatetime型に変換
df = df.with_columns(pl.col("transaction_date").str.to_datetime())

print(f"総データ件数: {len(df):,}件")
print(f"期間: {df['transaction_date'].min()} 〜 {df['transaction_date'].max()}")

総データ件数: 325,042件
期間: 2005-01-01 00:00:00 〜 2025-01-01 00:00:00


In [3]:
# 東京23区のリスト
tokyo_23_wards = [
    "千代田区",
    "中央区",
    "港区",
    "新宿区",
    "文京区",
    "台東区",
    "墨田区",
    "江東区",
    "品川区",
    "目黒区",
    "大田区",
    "世田谷区",
    "渋谷区",
    "中野区",
    "杉並区",
    "豊島区",
    "北区",
    "荒川区",
    "板橋区",
    "練馬区",
    "足立区",
    "葛飾区",
    "江戸川区",
]

# 23区データのフィルタリング
df_tokyo = df.filter(pl.col("Municipality").is_in(tokyo_23_wards))

print(f"東京23区のデータ件数: {len(df_tokyo):,}件")

東京23区のデータ件数: 325,042件


## 2. データ分割（学習期間 vs 予測期間）


In [13]:
# データを2005-2023年（学習）と2024-2025年（予測）に分割
train_df = df_tokyo.filter(pl.col("transaction_date") < pl.datetime(2024, 1, 1))
test_df = df_tokyo.filter(pl.col("transaction_date") >= pl.datetime(2024, 1, 1))

margin = 0.01
min_tsubo_price = train_df["tsubo_price"].quantile(margin)
max_tsubo_price = train_df["tsubo_price"].quantile(1 - margin)

train_df = train_df.filter(pl.col("tsubo_price").is_between(min_tsubo_price, max_tsubo_price))
test_df = test_df.filter(pl.col("tsubo_price").is_between(min_tsubo_price, max_tsubo_price))

print(f"学習データ: {len(train_df):,}件 ({train_df['transaction_date'].min()} - {train_df['transaction_date'].max()})")
print(f"テストデータ: {len(test_df):,}件 ({test_df['transaction_date'].min()} - {test_df['transaction_date'].max()})")

学習データ: 264,424件 (2005-01-01 00:00:00 - 2023-01-01 00:00:00)
テストデータ: 51,605件 (2024-01-01 00:00:00 - 2025-01-01 00:00:00)


In [ ]:
if False:
    df_tmp = train_df.with_columns(
        pl.col("tsubo_price").quantile(margin).alias("min_tsubo_price"),
        pl.col("tsubo_price").quantile(1 - margin).alias("max_tsubo_price"),
    ).filter(pl.col("tsubo_price").is_between(pl.col("min_tsubo_price"), pl.col("max_tsubo_price")))

    fig = go.Figure()

    # 除外前のデータ
    fig.add_trace(
        go.Histogram(x=train_df["tsubo_price"].to_list(), name="除外前 (Original)", marker_color="#E68B31", opacity=0.5)
    )

    # 除外後のデータ
    fig.add_trace(
        go.Histogram(x=df_tmp["tsubo_price"].to_list(), name="除外後 (Filtered)", marker_color="#337AB7", opacity=0.75)
    )

    # レイアウト設定
    fig.update_layout(
        title_text="外れ値除外の前後比較",
        xaxis_title_text="値",
        yaxis_title_text="頻度",
        barmode="overlay",  # 棒を重ねて表示
    )

    fig.show()

## 3. Prophet予測実行（区ごと）


In [54]:
# 主要な区を選択してProphet予測を実行
selected_wards_prophet = ["港区", "千代田区", "渋谷区", "世田谷区", "足立区", "葛飾区"]

param_grid = {
    "changepoint_prior_scale": [0.001, 0.05, 0.5],
    "seasonality_prior_scale": [0.01, 1.0, 10.0],
}

prophet_results = {}
desc_results = {}
for ward in selected_wards_prophet:
    print(f"\n{'=' * 60}")
    print(f"{ward} - Prophet予測")
    print("=" * 60)

    # 学習データとテストデータを区で絞り込み
    train_ward = train_df.filter(pl.col("Municipality") == ward)
    test_ward = test_df.filter(pl.col("Municipality") == ward)
    all_ward = pl.concat([train_ward, test_ward])

    # 月単位で集約（学習データ）
    train_monthly = (
        train_ward.group_by_dynamic("transaction_date", every="1mo")
        .agg(pl.col("tsubo_price").mean().alias("y"))
        .sort("transaction_date")
        .rename({"transaction_date": "ds"})
    )

    # Polars -> Pandas変換
    train_prophet = train_monthly.to_pandas()

    print(f"学習データ: {len(train_prophet)}月分")

    # Prophetモデル作成
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.001,
        seasonality_prior_scale=0.1,
    )

    # 学習
    model.fit(train_prophet)

    # テストデータの予測
    # データポイントレベルで予測するため、test_wardの日付を使用
    test_dates = pd.DataFrame({"ds": test_ward["transaction_date"].to_pandas()})
    forecast = model.predict(test_dates)

    # 実測値と予測値を比較
    y_true = test_ward["tsubo_price"].to_numpy()
    y_pred = forecast["yhat"].values

    # 評価指標
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"MAE: {mae:,.0f}円/坪")
    print(f"MAPE: {mape:.2f}%")

    # 結果を保存
    prophet_results[ward] = {
        "mae": mae,
        "mape": mape,
        "y_true": y_true,
        "y_pred": y_pred,
        "test_dates": test_ward["transaction_date"].to_list(),
        "forecast": forecast,
    }

    all_monthly = (
        all_ward.group_by_dynamic("transaction_date", every="1mo")
        .agg(pl.col("tsubo_price").mean().alias("y"))
        .sort("transaction_date")
        .rename({"transaction_date": "ds"})
    )
    all_prophet = all_monthly.to_pandas()
    forecast_all = model.predict(all_prophet)

    all_monthly_desc = (
        all_ward.group_by_dynamic("transaction_date", every="1mo")
        .agg(
            [
                pl.col("tsubo_price").mean().alias("mean_price"),
                pl.col("tsubo_price").median().alias("median_price"),
                pl.col("tsubo_price").count().alias("count"),
            ]
        )
        .sort("transaction_date")
        .rename({"transaction_date": "ds"})
    )
    all_monthly_desc = all_monthly_desc.join(pl.from_pandas(forecast_all), on="ds")
    desc_results[ward] = all_monthly_desc

print("\n" + "=" * 60)
print("全体サマリー")
print("=" * 60)
for ward, metrics in prophet_results.items():
    print(f"{ward:12s} - MAE: {metrics['mae']:>10,.0f}円/坪, MAPE: {metrics['mape']:>6.2f}%")

21:03:14 - cmdstanpy - INFO - Chain [1] start processing


21:03:14 - cmdstanpy - INFO - Chain [1] done processing



港区 - Prophet予測
学習データ: 19月分


21:03:15 - cmdstanpy - INFO - Chain [1] start processing
21:03:15 - cmdstanpy - INFO - Chain [1] done processing
21:03:15 - cmdstanpy - INFO - Chain [1] start processing


MAE: 1,047,443円/坪
MAPE: 26.95%

千代田区 - Prophet予測
学習データ: 19月分
MAE: 936,319円/坪
MAPE: 23.48%

渋谷区 - Prophet予測
学習データ: 19月分


...

MAE: 1,067,822円/坪
MAPE: 29.26%

世田谷区 - Prophet予測
学習データ: 19月分


21:03:15 - cmdstanpy - INFO - Chain [1] start processing
21:03:15 - cmdstanpy - INFO - Chain [1] done processing


MAE: 939,524円/坪
MAPE: 31.24%

足立区 - Prophet予測
学習データ: 19月分


21:03:16 - cmdstanpy - INFO - Chain [1] start processing
21:03:16 - cmdstanpy - INFO - Chain [1] done processing


MAE: 694,997円/坪
MAPE: 38.93%

葛飾区 - Prophet予測
学習データ: 19月分
MAE: 738,354円/坪
MAPE: 34.13%

全体サマリー
港区           - MAE:  1,047,443円/坪, MAPE:  26.95%
千代田区         - MAE:    936,319円/坪, MAPE:  23.48%
渋谷区          - MAE:  1,067,822円/坪, MAPE:  29.26%
世田谷区         - MAE:    939,524円/坪, MAPE:  31.24%
足立区          - MAE:    694,997円/坪, MAPE:  38.93%
葛飾区          - MAE:    738,354円/坪, MAPE:  34.13%


In [42]:
def plot_prophet_comparison(ward, df_desc):
    # グラフ作成
    fig = go.Figure()

    # 実績平均（学習期間）
    fig.add_trace(
        go.Scatter(
            x=df_desc["ds"].to_list(),
            y=df_desc["mean_price"].to_list(),
            mode="lines+markers",
            name="実績平均（学習期間）",
            line=dict(width=2, color="blue"),
            marker=dict(size=6),
        )
    )

    # 実績中央値（学習期間）
    fig.add_trace(
        go.Scatter(
            x=df_desc["ds"].to_list(),
            y=df_desc["median_price"].to_list(),
            mode="lines",
            name="実績中央値（学習期間）",
            line=dict(width=1, color="lightblue", dash="dot"),
            # visible="legendonly",
        )
    )

    # Prophet予測（全期間）
    fig.add_trace(
        go.Scatter(
            x=df_desc["ds"].to_list(),
            y=df_desc["yhat"].to_list(),
            mode="lines",
            name="Prophet予測",
            line=dict(width=2, color="red", dash="dash"),
        )
    )

    # Prophet信頼区間
    fig.add_trace(
        go.Scatter(
            x=df_desc["ds"].to_list(),
            y=df_desc["yhat_upper"].to_list(),
            mode="lines",
            name="予測信頼区間（上限）",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=df_desc["ds"].to_list(),
            y=df_desc["yhat_lower"].to_list(),
            mode="lines",
            name="予測信頼区間",
            line=dict(width=0),
            fillcolor="rgba(255, 0, 0, 0.2)",
            fill="tonexty",
            # visible="legendonly",
        )
    )

    # 垂直線（学習期間とテスト期間の境界）
    fig.add_vline(
        x=pd.Timestamp("2024-01-01").timestamp() * 1000,
        line_dash="dash",
        line_color="gray",
        annotation_text="2024/01/01 (予測開始)",
        annotation_position="top",
    )

    fig.update_layout(
        title=f"{ward} - 実績平均・中央値 vs Prophet予測（2005-2025年）",
        xaxis_title="年月",
        yaxis_title="坪単価（円/坪）",
        height=600,
        hovermode="x unified",
    )

    fig.show()

In [55]:
for ward, df_desc in desc_results.items():
    plot_prophet_comparison(ward, df_desc)

## 4. 可視化: 評価指標の比較


In [6]:
# 区ごとの評価指標の比較

wards_list = list(prophet_results.keys())
mae_list = [prophet_results[w]["mae"] for w in wards_list]
rmse_list = [prophet_results[w]["rmse"] for w in wards_list]
mape_list = [prophet_results[w]["mape"] for w in wards_list]

# サブプロットで3つの指標を表示
fig = make_subplots(rows=1, cols=3, subplot_titles=("MAE (円/坪)", "RMSE (円/坪)", "MAPE (%)"), horizontal_spacing=0.12)

# MAE
fig.add_trace(go.Bar(x=wards_list, y=mae_list, name="MAE", marker_color="lightblue"), row=1, col=1)

# RMSE
fig.add_trace(go.Bar(x=wards_list, y=rmse_list, name="RMSE", marker_color="lightcoral"), row=1, col=2)

# MAPE
fig.add_trace(go.Bar(x=wards_list, y=mape_list, name="MAPE", marker_color="lightgreen"), row=1, col=3)

fig.update_layout(title_text="Prophet予測精度の区ごと比較", height=500, showlegend=False)

fig.update_xaxes(tickangle=45)

fig.show()

## 7. 可視化: 実績平均・中央値 vs Prophet予測（時系列）

学習期間と予測期間を通して、実績の月次平均値・中央値とProphet予測値を比較します。


In [ ]:
# 区ごとに実績平均・中央値とProphet予測を時系列で比較

for ward in selected_wards_prophet:
    print(f"\n処理中: {ward}")

    # 全期間のデータを取得
    ward_all = df_tokyo.filter(pl.col("Municipality") == ward)

    # 月次集約（全期間）
    monthly_all = (
        ward_all.group_by_dynamic("transaction_date", every="1mo")
        .agg(
            [
                pl.col("tsubo_price").mean().alias("mean_price"),
                pl.col("tsubo_price").median().alias("median_price"),
                pl.col("tsubo_price").count().alias("count"),
            ]
        )
        .sort("transaction_date")
    )

    # 学習期間とテスト期間に分割
    monthly_train = monthly_all.filter(pl.col("transaction_date") < pl.datetime(2024, 1, 1))
    monthly_test = monthly_all.filter(pl.col("transaction_date") >= pl.datetime(2024, 1, 1))

    # Prophet予測（月次）
    # 学習期間のデータでProphetを学習
    train_prophet = monthly_train.select(
        [pl.col("transaction_date").alias("ds"), pl.col("mean_price").alias("y")]
    ).to_pandas()

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10.0,
    )
    model.fit(train_prophet)

    # 全期間の予測（学習期間+テスト期間）
    future = model.make_future_dataframe(periods=len(monthly_test), freq="MS")
    forecast = model.predict(future)

    # Polars DataFrameに変換
    forecast_pl = pl.from_pandas(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]])
    break

20:39:22 - cmdstanpy - INFO - Chain [1] start processing
20:39:22 - cmdstanpy - INFO - Chain [1] done processing



処理中: 港区
